# IND320 Course Project, Part 3

## Code access and direct links

- The project is deployed here: [ind320-henrikengdal-project](https://ind320-henrikengdal-project.streamlit.app/)
- The code is accessible at the repository: [henrikengdal/ind320-henrikengdal-project](https://github.com/HenrikEngd/IND320-HenrikEngdal-Project.git)

## AI Usage

AI plays a multifaceted role throughout this project, primarily serving as an assistant and analytical tool. The project leverages AI in several areas:

**Development and Code Generation:**
AI assists in writing and optimizing code for the application, including API integration and data processing functions.

**Data Analysis and Insights:**
AI helps analyze data patterns and identifying trends. It assists in generating meaningful statistical summaries and suggesting appropriate visualization techniques for the given data.

**Documentation and Communication:**
AI supports the creation of clear documentation, such as code comments, and user interface text. It helps structure the project documentation and ensures technical concepts are communicated effectively.

**Problem-Solving and Debugging:**
Throughout the development process, AI serves as a coding companion, helping troubleshoot issues, optimize data processing workflows, and suggesting best practices for API usage.

## Import Required Libraries

In [1]:
import pandas as pd
import requests
from datetime import datetime

/Users/henrikengdal/Documents/GitHub/IND320-henrikengdal-project/.conda/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## Norwegian Electricity Price Areas

Norway is divided into 5 electricity price areas. Here we create a DataFrame with representative cities for each price area along with their geographical coordinates (longitude and latitude).

In [2]:
# Create DataFrame with Norwegian cities representing the 5 electricity price areas
# NO1: Oslo (Eastern Norway)
# NO2: Kristiansand (Southern Norway)
# NO3: Trondheim (Central Norway)
# NO4: Tromsø (Northern Norway)
# NO5: Bergen (Western Norway)

price_areas_data = {
    'price_area': ['NO1', 'NO2', 'NO3', 'NO4', 'NO5'],
    'city': ['Oslo', 'Kristiansand', 'Trondheim', 'Tromsø', 'Bergen'],
    'latitude': [59.9139, 58.1467, 63.4305, 69.6492, 60.3913],
    'longitude': [10.7522, 7.9956, 10.3951, 18.9553, 5.3221]
}

price_areas_df = pd.DataFrame(price_areas_data)
print("Norwegian Electricity Price Areas:")
price_areas_df

Norwegian Electricity Price Areas:


,price_area,city,latitude,longitude
0,NO1,Oslo,59.9139,10.7522
1,NO2,Kristiansand,58.1467,7.9956
2,NO3,Trondheim,63.4305,10.3951
3,NO4,Tromsø,69.6492,18.9553
4,NO5,Bergen,60.3913,5.3221


## Open-Meteo API Function

Create a function to download historical weather data from the Open-Meteo API using the ERA5 reanalysis model. The function will retrieve the same weather properties as in the CSV file from Part 1:
- Temperature at 2m
- Precipitation
- Wind speed at 10m
- Wind gusts at 10m
- Wind direction at 10m

In [3]:
def download_weather_data(latitude, longitude, year):
    """
    Download historical weather data from Open-Meteo API using ERA5 reanalysis model.
    
    Parameters:
    -----------
    latitude : float
        Latitude coordinate of the location
    longitude : float
        Longitude coordinate of the location
    year : int
        Year for which to download data
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame containing hourly weather data with columns:
        - time
        - temperature_2m (°C)
        - precipitation (mm)
        - wind_speed_10m (m/s)
        - wind_gusts_10m (m/s)
        - wind_direction_10m (°)
    """
    
    # Define date range for the year
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"
    
    # Open-Meteo API endpoint for historical weather (ERA5)
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    # Parameters matching the CSV file from Part 1
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "precipitation",
            "wind_speed_10m",
            "wind_gusts_10m",
            "wind_direction_10m"
        ],
        "timezone": "auto"
    }
    
    # Make API request
    print(f"Downloading weather data for coordinates ({latitude}, {longitude}) for year {year}...")
    response = requests.get(url, params=params)
    
    # Check if request was successful
    if response.status_code == 200:
        data = response.json()
        
        # Extract hourly data
        hourly_data = data['hourly']
        
        # Create DataFrame
        df = pd.DataFrame({
            'time': hourly_data['time'],
            'temperature_2m (°C)': hourly_data['temperature_2m'],
            'precipitation (mm)': hourly_data['precipitation'],
            'wind_speed_10m (m/s)': hourly_data['wind_speed_10m'],
            'wind_gusts_10m (m/s)': hourly_data['wind_gusts_10m'],
            'wind_direction_10m (°)': hourly_data['wind_direction_10m']
        })
        
        # Convert time column to datetime
        df['time'] = pd.to_datetime(df['time'])
        
        print(f"Successfully downloaded {len(df)} hourly records.")
        return df
    else:
        print(f"Error: API request failed with status code {response.status_code}")
        print(f"Response: {response.text}")
        return None

## Download Weather Data for Bergen (2019)

Now we'll apply the function to download weather data for Bergen for the year 2019.

In [4]:
# Get Bergen's coordinates from the price areas DataFrame
bergen_data = price_areas_df[price_areas_df['city'] == 'Bergen'].iloc[0]
bergen_lat = bergen_data['latitude']
bergen_lon = bergen_data['longitude']

print(f"Bergen coordinates: Latitude {bergen_lat}, Longitude {bergen_lon}")
print(f"Price area: {bergen_data['price_area']}")
print()

Bergen coordinates: Latitude 60.3913, Longitude 5.3221
Price area: NO5



In [5]:
# Download weather data for Bergen for 2019
bergen_2019_df = download_weather_data(bergen_lat, bergen_lon, 2019)

Successfully downloaded 8760 hourly records.
Successfully downloaded 8760 hourly records.


## Explore the Downloaded Data

Let's examine the structure and content of the downloaded weather data.

In [6]:
# Display first few rows
print("First 5 rows of Bergen 2019 weather data:")
bergen_2019_df.head()

First 5 rows of Bergen 2019 weather data:


,time,temperature_2m (°C),precipitation (mm),wind_speed_10m (m/s),wind_gusts_10m (m/s),wind_direction_10m (°)
0,2019-01-01 00:00:00,5.7,0.7,37.0,99.7,263
1,2019-01-01 01:00:00,5.8,0.2,41.0,107.3,278
2,2019-01-01 02:00:00,6.1,0.7,42.0,112.0,286
3,2019-01-01 03:00:00,6.3,0.5,40.9,105.8,298
4,2019-01-01 04:00:00,5.8,1.1,41.2,110.2,315


In [7]:
# Display data info
print("DataFrame Information:")
bergen_2019_df.info()

DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   time                    8760 non-null   datetime64[ns]
 1   temperature_2m (°C)     8760 non-null   float64       
 2   precipitation (mm)      8760 non-null   float64       
 3   wind_speed_10m (m/s)    8760 non-null   float64       
 4   wind_gusts_10m (m/s)    8760 non-null   float64       
 5   wind_direction_10m (°)  8760 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 410.8 KB


In [8]:
# Display summary statistics
print("Summary Statistics:")
bergen_2019_df.describe()

Summary Statistics:


,time,temperature_2m (°C),precipitation (mm),wind_speed_10m (m/s),wind_gusts_10m (m/s),wind_direction_10m (°)
count,8760,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000
mean,2019-07-02 11:30:00,7.831689,0.246518,10.688402,29.528185,186.517123
min,2019-01-01 00:00:00,-12.900000,0.000000,0.000000,2.500000,1.000000
25%,2019-04-02 05:45:00,3.100000,0.000000,6.100000,16.900000,117.000000
50%,2019-07-02 11:30:00,7.200000,0.000000,9.500000,26.600000,165.000000
75%,2019-10-01 17:15:00,12.100000,0.200000,14.300000,39.600000,279.000000
max,2019-12-31 23:00:00,31.700000,9.500000,46.700000,131.000000,360.000000
std,NaN,5.999381,0.583165,6.055750,16.107459,96.202031


## Verification

Let's verify that the data matches the expected format from the CSV file in Part 1.

In [9]:
# Check column names match the original CSV file
print("Columns in downloaded data:")
print(bergen_2019_df.columns.tolist())
print()

# Load original CSV for comparison
original_df = pd.read_csv('assets/open-meteo-subset.csv')
print("Columns in original CSV file:")
print(original_df.columns.tolist())
print()

# Check if column names match (except for time format)
match = set(bergen_2019_df.columns) == set(original_df.columns)
print(f"Column names match: {match}")

Columns in downloaded data:
['time', 'temperature_2m (°C)', 'precipitation (mm)', 'wind_speed_10m (m/s)', 'wind_gusts_10m (m/s)', 'wind_direction_10m (°)']

Columns in original CSV file:
['time', 'temperature_2m (°C)', 'precipitation (mm)', 'wind_speed_10m (m/s)', 'wind_gusts_10m (m/s)', 'wind_direction_10m (°)']

Column names match: True


## Summary

Successfully implemented:

1. **Price Areas DataFrame**: Created a pandas DataFrame containing the 5 Norwegian electricity price areas (NO1-NO5) with their representative cities and geographical coordinates:
   - NO1: Oslo (Eastern Norway)
   - NO2: Kristiansand (Southern Norway)
   - NO3: Trondheim (Central Norway)
   - NO4: Tromsø (Northern Norway)
   - NO5: Bergen (Western Norway)

2. **API Download Function**: Created a reusable function `download_weather_data()` that:
   - Takes latitude, longitude, and year as inputs
   - Downloads historical weather data from the Open-Meteo API using the ERA5 reanalysis model
   - Returns a pandas DataFrame with the same structure as the original CSV file
   - Includes all required weather properties: temperature, precipitation, wind speed, wind gusts, and wind direction

3. **Bergen 2019 Data**: Successfully downloaded 8,760 hourly weather records for Bergen for the year 2019, demonstrating that:
   - The API connection works correctly
   - The data format matches the original CSV file exactly
   - The function is ready to be used for other locations and years

The function can now be easily applied to download data for any of the other cities (Oslo, Kristiansand, Trondheim, Tromsø) or for different years as needed for further analysis.

## Optional: Download Data for All Cities

The function can be easily extended to download data for all 5 cities. Here's an example (commented out to avoid excessive API calls):

In [10]:
# Example: Download data for all cities (uncomment to run)
# This would download weather data for all 5 cities for a given year

"""
# Dictionary to store DataFrames for each city
all_cities_data = {}

# Year to download
year = 2019

# Loop through each city in the price areas DataFrame
for idx, row in price_areas_df.iterrows():
    city = row['city']
    lat = row['latitude']
    lon = row['longitude']
    
    print(f"\nDownloading data for {city}...")
    df = download_weather_data(lat, lon, year)
    
    if df is not None:
        # Store the DataFrame with city name as key
        all_cities_data[city] = df
        print(f"✓ Successfully downloaded data for {city}")

# Now you would have data for all 5 cities stored in all_cities_data dictionary
# Example: all_cities_data['Oslo'], all_cities_data['Bergen'], etc.
"""

print("Example code provided above (commented out to avoid excessive API calls)")

Example code provided above (commented out to avoid excessive API calls)


# CA3: Weather, Outliers, STL, and Spectrogram
This section adds:
- A DataFrame with Norwegian price areas (NO1–NO5) and representative cities with coordinates.
- A reusable Open‑Meteo ERA5 download function and a test download for Bergen (2019).
- Outlier/SPC analysis for temperature with DCT high‑pass seasonal adjustment.
- LOF anomaly detection for precipitation.
- STL decomposition and spectrogram utilities for Elhub production data.

At the end there is a small log cell to document work and AI assistance.

In [ ]:
# Price areas and representative cities
import pandas as pd

price_areas_data = {
    'price_area': ['NO1', 'NO2', 'NO3', 'NO4', 'NO5'],
    'city': ['Oslo', 'Kristiansand', 'Trondheim', 'Tromsø', 'Bergen'],
    # Approximate city centre coordinates
    'latitude': [59.9139, 58.1467, 63.4305, 69.6492, 60.3913],
    'longitude': [10.7522, 7.9956, 10.3951, 18.9553, 5.3221]
}
price_areas_df = pd.DataFrame(price_areas_data)
price_areas_df

In [ ]:
# Open-Meteo ERA5 download utility
import requests
from datetime import datetime

BASE_URL = "https://archive-api.open-meteo.com/v1/era5"  # Reanalysis endpoint

DEFAULT_HOURLY = [
    "temperature_2m","precipitation","wind_speed_10m","wind_gusts_10m","wind_direction_10m","relative_humidity_2m","surface_pressure","cloud_cover"
]

def fetch_open_meteo(lat: float, lon: float, year: int, hourly: list = None, tz: str = "UTC"):
    """Download hourly ERA5 reanalysis data for given coordinate and full year.
    Returns a pandas DataFrame with a datetime index.
    """
    if hourly is None:
        hourly = DEFAULT_HOURLY
    start = f"{year}-01-01"
    end = f"{year}-12-31"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "hourly": ",".join(hourly),
        "timezone": tz
    }
    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    # Build DataFrame
    times = data['hourly']['time']
    df = pd.DataFrame({'time': pd.to_datetime(times)})
    for var in hourly:
        if var in data['hourly']:
            df[var] = data['hourly'][var]
    return df

# Test download for Bergen 2019
bergen_lat, bergen_lon = 60.3913, 5.3221
bergen_2019_df = fetch_open_meteo(bergen_lat, bergen_lon, 2019)
bergen_2019_df.head()

In [ ]:
# Temperature SPC outlier detection using DCT-based seasonal adjustment
import numpy as np
import scipy.fftpack as fft
import plotly.graph_objects as go

def detect_temperature_outliers(df, temp_col="temperature_2m", time_col="time", dct_cutoff: int = 200, n_std: float = 3.5):
    """High-pass filter temperature via DCT, derive robust SPC bounds and plot original temperature
    with coloured outliers.
    Parameters
    ----------
    dct_cutoff : number of highest frequency coefficients to retain (acts as high-pass threshold)
    n_std : number of robust std (MAD*1.4826) from mean of SATV for control limits
    Returns: fig (plotly Figure), summary DataFrame of outliers
    """
    working = df.dropna(subset=[temp_col]).copy()
    temps = working[temp_col].values
    # DCT
    coeffs = fft.dct(temps, norm='ortho')
    # Zero out low frequency (seasonal) components except last dct_cutoff coefficients
    hp_coeffs = np.zeros_like(coeffs)
    hp_coeffs[-dct_cutoff:] = coeffs[-dct_cutoff:]
    satv = fft.idct(hp_coeffs, norm='ortho')  # seasonally adjusted temperature variation
    # Robust stats
    satv_mean = np.mean(satv)
    mad = np.median(np.abs(satv - np.median(satv)))
    robust_std = mad * 1.4826 if mad > 0 else np.std(satv)
    upper = satv_mean + n_std * robust_std
    lower = satv_mean - n_std * robust_std
    # Determine outliers via SATV bounds
    outlier_mask = (satv - satv_mean > n_std * robust_std) | (satv_mean - satv > n_std * robust_std)
    working['is_outlier'] = outlier_mask
    working['satv'] = satv
    # Plot
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=working[time_col], y=working[temp_col],
                             mode='lines', name='Temperature', line=dict(color='#1f77b4')))
    # Outliers overlay
    if outlier_mask.any():
        fig.add_trace(go.Scatter(x=working.loc[outlier_mask, time_col],
                                 y=working.loc[outlier_mask, temp_col], mode='markers',
                                 name='Outliers', marker=dict(color='red', size=6)))
    # Control limits shown as horizontal lines (same across time)
    fig.add_hline(y=np.mean(temps) + (upper - satv_mean), line_dash='dash', line_color='green', annotation_text='Upper CL')
    fig.add_hline(y=np.mean(temps) + (lower - satv_mean), line_dash='dash', line_color='orange', annotation_text='Lower CL')
    fig.update_layout(title='Temperature with SPC Outliers', xaxis_title='Time', yaxis_title='Temperature (°C)',
                      template='plotly_white', height=500)
    summary = working.loc[outlier_mask, [time_col, temp_col]].rename(columns={temp_col: 'temperature'})
    return fig, summary

# Test function on Bergen 2019
temp_fig, temp_outliers = detect_temperature_outliers(bergen_2019_df)
len(temp_outliers), temp_outliers.head()

In [ ]:
# Precipitation LOF anomaly detection
from sklearn.neighbors import LocalOutlierFactor

def detect_precip_lof(df, precip_col="precipitation", time_col="time", contamination: float = 0.01):
    working = df.dropna(subset=[precip_col]).copy()
    X = working[[precip_col]].values
    lof = LocalOutlierFactor(n_neighbors=35, contamination=contamination)
    labels = lof.fit_predict(X)  # -1 outlier
    working['is_anomaly'] = labels == -1
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=working[time_col], y=working[precip_col], mode='lines', name='Precipitation', line=dict(color='#2ca02c')))
    if working['is_anomaly'].any():
        fig.add_trace(go.Scatter(x=working.loc[working['is_anomaly'], time_col],
                                 y=working.loc[working['is_anomaly'], precip_col], mode='markers', name='Anomalies',
                                 marker=dict(color='purple', size=6)))
    fig.update_layout(title='Precipitation with LOF Anomalies', xaxis_title='Time', yaxis_title='Precipitation (mm)',
                      template='plotly_white', height=500)
    summary = working.loc[working['is_anomaly'], [time_col, precip_col]].rename(columns={precip_col: 'precipitation'})
    return fig, summary

# Test function
precip_fig, precip_anoms = detect_precip_lof(bergen_2019_df)
len(precip_anoms), precip_anoms.head()

In [ ]:
# STL decomposition for production data
from statsmodels.tsa.seasonal import STL
import json
import matplotlib.pyplot as plt

# Attempt to load production data (structure inferred: list of records with priceArea, productionGroup, startTime, quantityKwh)
with open('assets/production_data.json', 'r') as f:
    prod_raw = json.load(f)
prod_df = pd.DataFrame(prod_raw)
# Basic cleaning
prod_df['startTime'] = pd.to_datetime(prod_df['startTime'], errors='coerce')
prod_df = prod_df.dropna(subset=['startTime'])


def perform_stl(area='NO5', production_group='Hydro', period=24, seasonal=13, trend=25, robust=True):
    subset = prod_df[(prod_df['priceArea'] == area) & (prod_df['productionGroup'] == production_group)].copy()
    subset = subset.sort_values('startTime')
    y = subset['quantityKwh'].astype(float).values
    if len(y) < period * 2:
        raise ValueError('Not enough data points for STL with given period.')
    stl = STL(y, period=period, seasonal=seasonal, trend=trend, robust=robust)
    res = stl.fit()
    fig, axs = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
    axs[0].plot(subset['startTime'], y, label='Observed')
    axs[0].set_title('Observed')
    axs[1].plot(subset['startTime'], res.seasonal, label='Seasonal', color='orange')
    axs[1].set_title('Seasonal')
    axs[2].plot(subset['startTime'], res.trend, label='Trend', color='green')
    axs[2].set_title('Trend')
    axs[3].plot(subset['startTime'], res.resid, label='Residual', color='red')
    axs[3].set_title('Residual')
    for ax in axs:
        ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig

# Test STL
stl_fig = perform_stl()
stl_fig

In [ ]:
# Spectrogram of production time series
from scipy.signal import spectrogram

def create_spectrogram(area='NO5', production_group='Hydro', window_len=256, overlap=128):
    subset = prod_df[(prod_df['priceArea'] == area) & (prod_df['productionGroup'] == production_group)].copy()
    subset = subset.sort_values('startTime')
    y = subset['quantityKwh'].astype(float).values
    if len(y) < window_len:
        raise ValueError('Time series shorter than window length.')
    fs = 1.0  # hourly spacing => 1 sample/hour
    f, t, Sxx = spectrogram(y, fs=fs, nperseg=window_len, noverlap=overlap, scaling='spectrum')
    fig = go.Figure(data=go.Heatmap(x=t, y=f, z=10*np.log10(Sxx+1e-12), colorscale='Viridis'))
    fig.update_layout(title='Spectrogram (dB)', xaxis_title='Time (hours offset)', yaxis_title='Frequency (cycles/hour)', height=500)
    return fig

# Test spectrogram
spec_fig = create_spectrogram()
spec_fig

### Work Log & AI Assistance
Fill in details about what was done manually vs assisted by AI, decisions on parameter defaults (DCT cutoff=200, n_std=3.5, LOF contamination=1%, STL period=24, seasonal window=13, trend window=25, spectrogram window=256/overlap=128).